In [ ]:
import pandas as pd

data = pd.read_csv('filtered_security_commits.csv')
data.info()

In [ ]:
df_filtered = data[data['repo'] == 'django/django'].copy()

# Display the first few rows of the filtered DataFrame to verify
print("Filtered DataFrame (df_filtered) head:")
print(df_filtered.head())

# Display information about the filtered DataFrame
print("\nFiltered DataFrame (df_filtered) info:")
df_filtered.info()

In [ ]:
df_filtered.to_csv('django_security_commits.csv', index=False)

In [ ]:
import os
import re
import csv
import time
import ast
import requests
from git import Repo, GitCommandError

# ─── CONFIG ────────────────────────────────────────────────────────────────────

INPUT_CSV    = "django_security_commits.csv"
WORKDIR      = "./cloned_repos/"
OUTPUT_CSV   = "ground_truth_functions_for_django.csv"

CVE_REGEX    = re.compile(r"(CVE-\d{4}-\d{4,7})", re.IGNORECASE)
HUNK_HDR_RE  = re.compile(r"^@@ -(\d+)(?:,\d+)? \+(\d+)(?:,\d+)? @@")
NVD_API_URL    = "https://services.nvd.nist.gov/rest/json/cve/1.0/{}"
NVD_RATE_LIMIT = 6  # per second

# ─── HELPERS ──────────────────────────────────────────────────────────────────

def ensure_repo(full_name):
    owner, name = full_name.split("/")
    path = os.path.join(WORKDIR, name)
    url  = f"https://github.com/{full_name}.git"
    if not os.path.isdir(path):
        print(f"[+] Cloning {full_name}…")
        Repo.clone_from(url, path)
    else:
        try:
            print(f"[+] Updating {full_name}…")
            Repo(path).remote().pull()
        except GitCommandError:
            print("    (pull failed, using existing copy)")
    return Repo(path)

def get_file_content(repo, sha, file_path):
    try:
        return repo.git.show(f"{sha}:{file_path}")
    except Exception as e:
        print(f"    ! could not show {sha}:{file_path}: {e}")
        return ""

def first_hunk_line_numbers(repo, sha, file_path):
    commit = repo.commit(sha)
    parent = commit.parents[0] if commit.parents else None
    if not parent:
        return None, None

    # get the patch for this file
    for d in commit.diff(parent, create_patch=True):
        if d.a_path != file_path:
            continue
        patch = d.diff.decode("utf-8", errors="ignore")
        for line in patch.splitlines():
            m = HUNK_HDR_RE.match(line)
            if m:
                old_start = int(m.group(1))
                new_start = int(m.group(2))
                return old_start, new_start
        print(f"    (no hunk header found in patch of {file_path})")
        return None, None
    print(f"    (file {file_path} not found in diff of {sha})")
    return None, None

def extract_enclosing_function(src, target_line):
    try:
        tree = ast.parse(src)
    except Exception:
        return ""
    candidate = None
    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef) and hasattr(node, "end_lineno"):
            if node.lineno <= target_line <= node.end_lineno:
                span = node.end_lineno - node.lineno
                if candidate is None or span < candidate[0]:
                    candidate = (span, node)
    if not candidate:
        return ""
    _, fn = candidate
    seg = ast.get_source_segment(src, fn)
    if seg:
        return seg
    lines = src.splitlines()
    return "\n".join(lines[fn.lineno - 1 : fn.end_lineno])

def fetch_cwes_for_cve(cve):
    try:
        r = requests.get(NVD_API_URL.format(cve), timeout=10)
        data = r.json()
        items = data["result"]["CVE_Items"]
        cwes = {desc["value"]
                for pt in items[0]["cve"]["problemtype"]["problemtype_data"]
                for desc in pt["description"]
                if desc["value"].startswith("CWE-")}
        return sorted(cwes) or ["UNKNOWN"]
    except Exception as e:
        print(f"    ! NVD lookup failed for {cve}: {e}")
        return ["UNKNOWN"]
    finally:
        time.sleep(1.0 / NVD_RATE_LIMIT)

# ─── MAIN ────────────────────────────────────────────────────────────────────

def main():
    os.makedirs(WORKDIR, exist_ok=True)
    with open(INPUT_CSV, newline="", encoding="utf-8") as fin, \
         open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as fout:

        rdr = csv.DictReader(fin)
        wtr = csv.DictWriter(fout, fieldnames=[
            "repo","sha","file_path","cve_ids","cwe_ids",
            "vulnerable_function_source","patched_function_source"
        ], quoting=csv.QUOTE_ALL)
        wtr.writeheader()

        for row in rdr:
            repo_name = row["repo"]
            sha       = row["sha"]
            msg       = row.get("message","")
            print(f"\n>> Processing {repo_name}@{sha[:7]}")

            # CVE parsing
            cves = sorted({c.upper() for c in CVE_REGEX.findall(msg)}) or ["UNKNOWN"]
            cwe_set = set()
            for cve in cves:
                if cve != "UNKNOWN":
                    cwe_set.update(fetch_cwes_for_cve(cve))
            if not cwe_set:
                cwe_set = {"UNKNOWN"}

            repo = ensure_repo(repo_name)
            commit = repo.commit(sha)
            parent = commit.parents[0].hexsha if commit.parents else None
            if not parent:
                print("    ! no parent commit, skipping")
                continue

            # walk changed python files
            for diff in commit.diff(parent, create_patch=True):
                path = diff.a_path
                if not path or not path.endswith(".py"):
                    continue
                print(f"   - file {path}")

                rem, add = first_hunk_line_numbers(repo, sha, path)
                print(f"      removed_line={rem}, added_line={add}")

                old_txt = get_file_content(repo, parent, path)
                new_txt = get_file_content(repo, sha,    path)

                vul_fn = extract_enclosing_function(old_txt, rem) if rem else ""
                pat_fn = extract_enclosing_function(new_txt, add) if add else ""

                wtr.writerow({
                    "repo":                        repo_name,
                    "sha":                         sha,
                    "file_path":                   path,
                    "cve_ids":                     ",".join(cves),
                    "cwe_ids":                     ",".join(sorted(cwe_set)),
                    "vulnerable_function_source":  vul_fn,
                    "patched_function_source":     pat_fn
                })

    print(f"\n Done! Output in {OUTPUT_CSV}")

if __name__ == "__main__":
    main()


In [ ]:
df_filtered = data[data['repo'] == 'ansible/ansible'].copy()

# Display the first few rows of the filtered DataFrame to verify
print("Filtered DataFrame (df_filtered) head:")
print(df_filtered.head())

# Display information about the filtered DataFrame
print("\nFiltered DataFrame (df_filtered) info:")
df_filtered.info()

In [ ]:
df_filtered.to_csv('ansible_security_commits.csv', index=False)
df_filtered.info()

In [ ]:
import os
import re
import csv
import time
import ast
import requests
from git import Repo, GitCommandError



INPUT_CSV    = "ansible_security_commits.csv"
WORKDIR      = "./cloned_repos/"
OUTPUT_CSV   = "ground_truth_functions_for_ansible.csv"

CVE_REGEX    = re.compile(r"(CVE-\d{4}-\d{4,7})", re.IGNORECASE)
HUNK_HDR_RE  = re.compile(r"^@@ -(\d+)(?:,\d+)? \+(\d+)(?:,\d+)? @@")
NVD_API_URL    = "https://services.nvd.nist.gov/rest/json/cve/1.0/{}"
NVD_RATE_LIMIT = 6  # per second


def ensure_repo(full_name):
    owner, name = full_name.split("/")
    path = os.path.join(WORKDIR, name)
    url  = f"https://github.com/{full_name}.git"
    if not os.path.isdir(path):
        print(f"[+] Cloning {full_name}…")
        Repo.clone_from(url, path)
    else:
        try:
            print(f"[+] Updating {full_name}…")
            Repo(path).remote().pull()
        except GitCommandError:
            print("    (pull failed, using existing copy)")
    return Repo(path)

def get_file_content(repo, sha, file_path):
    try:
        return repo.git.show(f"{sha}:{file_path}")
    except Exception as e:
        print(f"    ! could not show {sha}:{file_path}: {e}")
        return ""

def first_hunk_line_numbers(repo, sha, file_path):
    commit = repo.commit(sha)
    parent = commit.parents[0] if commit.parents else None
    if not parent:
        return None, None

    # get the patch for this file
    for d in commit.diff(parent, create_patch=True):
        if d.a_path != file_path:
            continue
        patch = d.diff.decode("utf-8", errors="ignore")
        for line in patch.splitlines():
            m = HUNK_HDR_RE.match(line)
            if m:
                old_start = int(m.group(1))
                new_start = int(m.group(2))
                return old_start, new_start
        print(f"    (no hunk header found in patch of {file_path})")
        return None, None
    print(f"    (file {file_path} not found in diff of {sha})")
    return None, None

def extract_enclosing_function(src, target_line):
    try:
        tree = ast.parse(src)
    except Exception:
        return ""
    candidate = None
    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef) and hasattr(node, "end_lineno"):
            if node.lineno <= target_line <= node.end_lineno:
                span = node.end_lineno - node.lineno
                if candidate is None or span < candidate[0]:
                    candidate = (span, node)
    if not candidate:
        return ""
    _, fn = candidate
    seg = ast.get_source_segment(src, fn)
    if seg:
        return seg
    lines = src.splitlines()
    return "\n".join(lines[fn.lineno - 1 : fn.end_lineno])

def fetch_cwes_for_cve(cve):
    try:
        r = requests.get(NVD_API_URL.format(cve), timeout=10)
        data = r.json()
        items = data["result"]["CVE_Items"]
        cwes = {desc["value"]
                for pt in items[0]["cve"]["problemtype"]["problemtype_data"]
                for desc in pt["description"]
                if desc["value"].startswith("CWE-")}
        return sorted(cwes) or ["UNKNOWN"]
    except Exception as e:
        print(f"    ! NVD lookup failed for {cve}: {e}")
        return ["UNKNOWN"]
    finally:
        time.sleep(1.0 / NVD_RATE_LIMIT)



def main():
    os.makedirs(WORKDIR, exist_ok=True)
    with open(INPUT_CSV, newline="", encoding="utf-8") as fin, \
         open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as fout:

        rdr = csv.DictReader(fin)
        wtr = csv.DictWriter(fout, fieldnames=[
            "repo","sha","file_path","cve_ids","cwe_ids",
            "vulnerable_function_source","patched_function_source"
        ], quoting=csv.QUOTE_ALL)
        wtr.writeheader()

        for row in rdr:
            repo_name = row["repo"]
            sha       = row["sha"]
            msg       = row.get("message","")
            print(f"\n>> Processing {repo_name}@{sha[:7]}")

            # CVE parsing
            cves = sorted({c.upper() for c in CVE_REGEX.findall(msg)}) or ["UNKNOWN"]
            cwe_set = set()
            for cve in cves:
                if cve != "UNKNOWN":
                    cwe_set.update(fetch_cwes_for_cve(cve))
            if not cwe_set:
                cwe_set = {"UNKNOWN"}

            repo = ensure_repo(repo_name)
            commit = repo.commit(sha)
            parent = commit.parents[0].hexsha if commit.parents else None
            if not parent:
                print("    ! no parent commit, skipping")
                continue

            # walk changed python files
            for diff in commit.diff(parent, create_patch=True):
                path = diff.a_path
                if not path or not path.endswith(".py"):
                    continue
                print(f"   - file {path}")

                rem, add = first_hunk_line_numbers(repo, sha, path)
                print(f"      removed_line={rem}, added_line={add}")

                old_txt = get_file_content(repo, parent, path)
                new_txt = get_file_content(repo, sha,    path)

                vul_fn = extract_enclosing_function(old_txt, rem) if rem else ""
                pat_fn = extract_enclosing_function(new_txt, add) if add else ""

                wtr.writerow({
                    "repo":                        repo_name,
                    "sha":                         sha,
                    "file_path":                   path,
                    "cve_ids":                     ",".join(cves),
                    "cwe_ids":                     ",".join(sorted(cwe_set)),
                    "vulnerable_function_source":  vul_fn,
                    "patched_function_source":     pat_fn
                })

    print(f"\n Done! Output in {OUTPUT_CSV}")

if __name__ == "__main__":
    main()


In [ ]:
import os
import re
import csv
import time
import ast
import requests
from git import Repo, GitCommandError



INPUT_CSV    = "airflow_security_commits.csv"
WORKDIR      = "./cloned_repos/"
OUTPUT_CSV   = "ground_truth_functions_for_airflow.csv"

CVE_REGEX    = re.compile(r"(CVE-\d{4}-\d{4,7})", re.IGNORECASE)
HUNK_HDR_RE  = re.compile(r"^@@ -(\d+)(?:,\d+)? \+(\d+)(?:,\d+)? @@")
NVD_API_URL    = "https://services.nvd.nist.gov/rest/json/cve/1.0/{}"
NVD_RATE_LIMIT = 6  # per second


def ensure_repo(full_name):
    owner, name = full_name.split("/")
    path = os.path.join(WORKDIR, name)
    url  = f"https://github.com/{full_name}.git"
    if not os.path.isdir(path):
        print(f"[+] Cloning {full_name}…")
        Repo.clone_from(url, path)
    else:
        try:
            print(f"[+] Updating {full_name}…")
            Repo(path).remote().pull()
        except GitCommandError:
            print("    (pull failed, using existing copy)")
    return Repo(path)

def get_file_content(repo, sha, file_path):
    try:
        return repo.git.show(f"{sha}:{file_path}")
    except Exception as e:
        print(f"    ! could not show {sha}:{file_path}: {e}")
        return ""

def first_hunk_line_numbers(repo, sha, file_path):
    commit = repo.commit(sha)
    parent = commit.parents[0] if commit.parents else None
    if not parent:
        return None, None

    # get the patch for this file
    for d in commit.diff(parent, create_patch=True):
        if d.a_path != file_path:
            continue
        patch = d.diff.decode("utf-8", errors="ignore")
        for line in patch.splitlines():
            m = HUNK_HDR_RE.match(line)
            if m:
                old_start = int(m.group(1))
                new_start = int(m.group(2))
                return old_start, new_start
        print(f"    (no hunk header found in patch of {file_path})")
        return None, None
    print(f"    (file {file_path} not found in diff of {sha})")
    return None, None

def extract_enclosing_function(src, target_line):
    try:
        tree = ast.parse(src)
    except Exception:
        return ""
    candidate = None
    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef) and hasattr(node, "end_lineno"):
            if node.lineno <= target_line <= node.end_lineno:
                span = node.end_lineno - node.lineno
                if candidate is None or span < candidate[0]:
                    candidate = (span, node)
    if not candidate:
        return ""
    _, fn = candidate
    seg = ast.get_source_segment(src, fn)
    if seg:
        return seg
    lines = src.splitlines()
    return "\n".join(lines[fn.lineno - 1 : fn.end_lineno])

def fetch_cwes_for_cve(cve):
    try:
        r = requests.get(NVD_API_URL.format(cve), timeout=10)
        data = r.json()
        items = data["result"]["CVE_Items"]
        cwes = {desc["value"]
                for pt in items[0]["cve"]["problemtype"]["problemtype_data"]
                for desc in pt["description"]
                if desc["value"].startswith("CWE-")}
        return sorted(cwes) or ["UNKNOWN"]
    except Exception as e:
        print(f"    ! NVD lookup failed for {cve}: {e}")
        return ["UNKNOWN"]
    finally:
        time.sleep(1.0 / NVD_RATE_LIMIT)



def main():
    os.makedirs(WORKDIR, exist_ok=True)
    with open(INPUT_CSV, newline="", encoding="utf-8") as fin, \
         open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as fout:

        rdr = csv.DictReader(fin)
        wtr = csv.DictWriter(fout, fieldnames=[
            "repo","sha","file_path","cve_ids","cwe_ids",
            "vulnerable_function_source","patched_function_source"
        ], quoting=csv.QUOTE_ALL)
        wtr.writeheader()

        for row in rdr:
            repo_name = row["repo"]
            sha       = row["sha"]
            msg       = row.get("message","")
            print(f"\n>> Processing {repo_name}@{sha[:7]}")

            # CVE parsing
            cves = sorted({c.upper() for c in CVE_REGEX.findall(msg)}) or ["UNKNOWN"]
            cwe_set = set()
            for cve in cves:
                if cve != "UNKNOWN":
                    cwe_set.update(fetch_cwes_for_cve(cve))
            if not cwe_set:
                cwe_set = {"UNKNOWN"}

            repo = ensure_repo(repo_name)
            commit = repo.commit(sha)
            parent = commit.parents[0].hexsha if commit.parents else None
            if not parent:
                print("    ! no parent commit, skipping")
                continue

            # walk changed python files
            for diff in commit.diff(parent, create_patch=True):
                path = diff.a_path
                if not path or not path.endswith(".py"):
                    continue
                print(f"   - file {path}")

                rem, add = first_hunk_line_numbers(repo, sha, path)
                print(f"      removed_line={rem}, added_line={add}")

                old_txt = get_file_content(repo, parent, path)
                new_txt = get_file_content(repo, sha,    path)

                vul_fn = extract_enclosing_function(old_txt, rem) if rem else ""
                pat_fn = extract_enclosing_function(new_txt, add) if add else ""

                wtr.writerow({
                    "repo":                        repo_name,
                    "sha":                         sha,
                    "file_path":                   path,
                    "cve_ids":                     ",".join(cves),
                    "cwe_ids":                     ",".join(sorted(cwe_set)),
                    "vulnerable_function_source":  vul_fn,
                    "patched_function_source":     pat_fn
                })

    print(f"\n Done! Output in {OUTPUT_CSV}")

if __name__ == "__main__":
    main()


In [ ]:
df_filtered = data[data['repo'] == 'paramiko/paramiko'].copy()

# Display the first few rows of the filtered DataFrame to verify
print("Filtered DataFrame (df_filtered) head:")
print(df_filtered.head())

# Display information about the filtered DataFrame
print("\nFiltered DataFrame (df_filtered) info:")
df_filtered.info()

In [ ]:
df_filtered.to_csv('paramiko_security_commits.csv', index=False)

In [ ]:
import os
import re
import csv
import time
import ast
import requests
from git import Repo, GitCommandError

# ─── CONFIG ────────────────────────────────────────────────────────────────────

INPUT_CSV    = "paramiko_security_commits.csv"
WORKDIR      = "./cloned_repos/"
OUTPUT_CSV   = "ground_truth_functions_for_paramiko.csv"

CVE_REGEX    = re.compile(r"(CVE-\d{4}-\d{4,7})", re.IGNORECASE)
HUNK_HDR_RE  = re.compile(r"^@@ -(\d+)(?:,\d+)? \+(\d+)(?:,\d+)? @@")
NVD_API_URL    = "https://services.nvd.nist.gov/rest/json/cve/1.0/{}"
NVD_RATE_LIMIT = 6  # per second

# ─── HELPERS ──────────────────────────────────────────────────────────────────

def ensure_repo(full_name):
    owner, name = full_name.split("/")
    path = os.path.join(WORKDIR, name)
    url  = f"https://github.com/{full_name}.git"
    if not os.path.isdir(path):
        print(f"[+] Cloning {full_name}…")
        Repo.clone_from(url, path)
    else:
        try:
            print(f"[+] Updating {full_name}…")
            Repo(path).remote().pull()
        except GitCommandError:
            print("    (pull failed, using existing copy)")
    return Repo(path)

def get_file_content(repo, sha, file_path):
    try:
        return repo.git.show(f"{sha}:{file_path}")
    except Exception as e:
        print(f"    ! could not show {sha}:{file_path}: {e}")
        return ""

def first_hunk_line_numbers(repo, sha, file_path):
    commit = repo.commit(sha)
    parent = commit.parents[0] if commit.parents else None
    if not parent:
        return None, None

    # get the patch for this file
    for d in commit.diff(parent, create_patch=True):
        if d.a_path != file_path:
            continue
        patch = d.diff.decode("utf-8", errors="ignore")
        for line in patch.splitlines():
            m = HUNK_HDR_RE.match(line)
            if m:
                old_start = int(m.group(1))
                new_start = int(m.group(2))
                return old_start, new_start
        print(f"    (no hunk header found in patch of {file_path})")
        return None, None
    print(f"    (file {file_path} not found in diff of {sha})")
    return None, None

def extract_enclosing_function(src, target_line):
    try:
        tree = ast.parse(src)
    except Exception:
        return ""
    candidate = None
    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef) and hasattr(node, "end_lineno"):
            if node.lineno <= target_line <= node.end_lineno:
                span = node.end_lineno - node.lineno
                if candidate is None or span < candidate[0]:
                    candidate = (span, node)
    if not candidate:
        return ""
    _, fn = candidate
    seg = ast.get_source_segment(src, fn)
    if seg:
        return seg
    lines = src.splitlines()
    return "\n".join(lines[fn.lineno - 1 : fn.end_lineno])

def fetch_cwes_for_cve(cve):
    try:
        r = requests.get(NVD_API_URL.format(cve), timeout=10)
        data = r.json()
        items = data["result"]["CVE_Items"]
        cwes = {desc["value"]
                for pt in items[0]["cve"]["problemtype"]["problemtype_data"]
                for desc in pt["description"]
                if desc["value"].startswith("CWE-")}
        return sorted(cwes) or ["UNKNOWN"]
    except Exception as e:
        print(f"    ! NVD lookup failed for {cve}: {e}")
        return ["UNKNOWN"]
    finally:
        time.sleep(1.0 / NVD_RATE_LIMIT)

# ─── MAIN ────────────────────────────────────────────────────────────────────

def main():
    os.makedirs(WORKDIR, exist_ok=True)
    with open(INPUT_CSV, newline="", encoding="utf-8") as fin, \
         open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as fout:

        rdr = csv.DictReader(fin)
        wtr = csv.DictWriter(fout, fieldnames=[
            "repo","sha","file_path","cve_ids","cwe_ids",
            "vulnerable_function_source","patched_function_source"
        ], quoting=csv.QUOTE_ALL)
        wtr.writeheader()

        for row in rdr:
            repo_name = row["repo"]
            sha       = row["sha"]
            msg       = row.get("message","")
            print(f"\n>> Processing {repo_name}@{sha[:7]}")

            # CVE parsing
            cves = sorted({c.upper() for c in CVE_REGEX.findall(msg)}) or ["UNKNOWN"]
            cwe_set = set()
            for cve in cves:
                if cve != "UNKNOWN":
                    cwe_set.update(fetch_cwes_for_cve(cve))
            if not cwe_set:
                cwe_set = {"UNKNOWN"}

            repo = ensure_repo(repo_name)
            commit = repo.commit(sha)
            parent = commit.parents[0].hexsha if commit.parents else None
            if not parent:
                print("    ! no parent commit, skipping")
                continue

            # walk changed python files
            for diff in commit.diff(parent, create_patch=True):
                path = diff.a_path
                if not path or not path.endswith(".py"):
                    continue
                print(f"   - file {path}")

                rem, add = first_hunk_line_numbers(repo, sha, path)
                print(f"      removed_line={rem}, added_line={add}")

                old_txt = get_file_content(repo, parent, path)
                new_txt = get_file_content(repo, sha,    path)

                vul_fn = extract_enclosing_function(old_txt, rem) if rem else ""
                pat_fn = extract_enclosing_function(new_txt, add) if add else ""

                wtr.writerow({
                    "repo":                        repo_name,
                    "sha":                         sha,
                    "file_path":                   path,
                    "cve_ids":                     ",".join(cves),
                    "cwe_ids":                     ",".join(sorted(cwe_set)),
                    "vulnerable_function_source":  vul_fn,
                    "patched_function_source":     pat_fn
                })

    print(f"\n Done! Output in {OUTPUT_CSV}")

if __name__ == "__main__":
    main()
